# Experiment 10: FIND-S Algorithm

**Objective:** Implement the FIND-S concept learning algorithm from machine learning to find the maximally specific hypothesis consistent with positive training examples.


In [ ]:
import pandas as pd
import os

# 1. Generate sample play_tennis.csv if not exists
sample = """Outlook,Temperature,Humidity,Wind,PlayTennis
Sunny,Hot,High,Weak,No
Sunny,Hot,High,Strong,No
Overcast,Hot,High,Weak,Yes
Rain,Mild,High,Weak,Yes
Rain,Cool,Normal,Weak,Yes
Rain,Cool,Normal,Strong,No
Overcast,Cool,Normal,Strong,Yes
Sunny,Mild,High,Weak,No
Sunny,Cool,Normal,Weak,Yes
Rain,Mild,Normal,Weak,Yes
Sunny,Mild,Normal,Strong,Yes
Overcast,Mild,High,Strong,Yes
Overcast,Hot,Normal,Weak,Yes
Rain,Mild,High,Strong,No
"""
with open('play_tennis.csv', 'w', newline='') as f:
    f.write(sample)

df = pd.read_csv('play_tennis.csv')
print("Dataset loaded:")
df


In [ ]:
# 2. Define FIND-S Function
def find_s(df, target_col, positive_label=None, verbose=True):
    df = df.copy()
    if positive_label is None:
        unique = df[target_col].unique().tolist()
        candidates = [u for u in unique if str(u).strip().lower() in ('yes', 'y', 'positive', '1', 'true', 't')]
        positive_label = candidates[0] if candidates else unique[0]
        
    features = [c for c in df.columns if c != target_col]
    h = [None] * len(features)  # None means Ø (most specific)
    steps = [("Initial", h.copy())]
    
    if verbose:
        print("Initial hypothesis:", ["Ø" for _ in features])
        
    for idx, row in df.iterrows():
        row_label = str(row[target_col]).strip().lower()
        if row_label == str(positive_label).strip().lower():
            for i, feat in enumerate(features):
                value = row[feat]
                if h[i] is None:
                    h[i] = value
                elif h[i] != value:
                    h[i] = '?'
            steps.append((f"After positive example row {idx+1}", h.copy()))
            if verbose:
                readable = [(h_i if h_i is not None else 'Ø') for h_i in h]
                print(f"After positive example {idx+1}: {[row[f] for f in features]} -> hypothesis: {readable}")
                
    return features, h, positive_label, steps


In [ ]:
# 3. Define Prediction and Evaluation Functions
def predict_with_h(h, features, instance):
    for i, feat in enumerate(features):
        constraint = h[i]
        val = instance[feat]
        if constraint == '?':
            continue
        if constraint is None or constraint != val:
            return False
    return True

def evaluate_hypothesis(h, features, df, target_col, positive_label=None):
    if positive_label is None:
        unique = df[target_col].unique().tolist()
        candidates = [u for u in unique if str(u).strip().lower() in ('yes', 'y', 'positive', '1', 'true', 't')]
        positive_label = candidates[0] if candidates else unique[0]
        
    preds = []
    TP = TN = FP = FN = 0
    for _, row in df.iterrows():
        pred_pos = predict_with_h(h, features, row)
        actual_pos = (str(row[target_col]).strip().lower() == str(positive_label).strip().lower())
        preds.append('Yes' if pred_pos else 'No')
        if pred_pos and actual_pos:
            TP += 1
        elif pred_pos and not actual_pos:
            FP += 1
        elif not pred_pos and actual_pos:
            FN += 1
        else:
            TN += 1
            
    total = len(df)
    accuracy = (TP + TN) / total if total > 0 else 0.0
    return preds, accuracy, TP, TN, FP, FN


In [ ]:
# 4. Run FIND-S on dataset
features, h, pos_label, _ = find_s(df, target_col="PlayTennis", verbose=True)

print("
Final hypothesis (feature -> constraint):")
readable = [(h_i if h_i is not None else 'Ø') for h_i in h]
for feat, constraint in zip(features, readable):
    print(f"  {feat} -> {constraint}")
    
preds, acc, TP, TN, FP, FN = evaluate_hypothesis(h, features, df, target_col="PlayTennis")
print("
Evaluation on dataset:")
print(f"  Positive label used: {pos_label}")
print(f"  Accuracy: {acc:.3f} (TP={TP}, TN={TN}, FP={FP}, FN={FN})")
